# Ensemble Learning Benchmark

## Setup

In [ ]:
REPO_URL = "https://github.com/rayyantaimoor1/ensemble-benchmark.git"
import os
if not os.path.exists("ensemble-benchmark"):
    !git clone {REPO_URL}
%cd ensemble-benchmark

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("src"))
import warnings
warnings.filterwarnings("ignore")

FAST_MODE = True

METRICS_DIR = "results/metrics"
os.makedirs(METRICS_DIR, exist_ok=True)

from data_pipeline import load_raw_data, make_splits
from models import get_baseline_models as _get_baseline_models
from benchmark import run_full_benchmark, run_scalability_sweep, get_learning_curve, get_feature_importances, run_kfold_cv
from tuning import tune_all_models
from plots import plot_speed_vs_performance, plot_scalability_curves, plot_loss_convergence, plot_feature_importance_divergence, plot_kfold_cv

def get_baseline_models():
    models = _get_baseline_models()
    if FAST_MODE:
        for m in models.values():
            if hasattr(m, "n_estimators"):
                m.set_params(n_estimators=60)
    return models

## Data

In [ ]:
df = load_raw_data()
df.shape

In [ ]:
splits = make_splits(df)
X_train, y_train = splits["X_train"], splits["y_train"]
X_val, y_val = splits["X_val"], splits["y_val"]
X_test, y_test = splits["X_test"], splits["y_test"]
len(X_train), len(X_val), len(X_test)

## Baseline Benchmark

In [ ]:
baseline_models = get_baseline_models()
results_df = run_full_benchmark(baseline_models, X_train, y_train, X_test, y_test)
results_df.to_csv(f"{METRICS_DIR}/full_benchmark_results.csv", index=False)
results_df

## Hyperparameter Tuning

In [ ]:
RUN_TUNING = False

if RUN_TUNING:
    tuned_models, best_params = tune_all_models(X_train, y_train, n_iter=20, cv=3)
    models_for_analysis = tuned_models
else:
    models_for_analysis = get_baseline_models()

## K-Fold Cross-Validation

In [ ]:
N_SPLITS = 5

cv_models = get_baseline_models()
cv_df = run_kfold_cv(cv_models, X_train, y_train, n_splits=N_SPLITS)
cv_df.to_csv(f"{METRICS_DIR}/kfold_cv_results.csv", index=False)
cv_df

In [ ]:
plot_kfold_cv(cv_df)

## Speed vs. Performance Frontier

In [ ]:
plot_speed_vs_performance(results_df)

## Scalability Curves

In [ ]:
sweep_sizes = (2_000, 8_000, 20_000, 50_000) if FAST_MODE else (10_000, 50_000, 100_000, 500_000)

sweep_df = run_scalability_sweep(get_baseline_models, X_train, y_train, X_test, y_test, row_sizes=sweep_sizes)
sweep_df.to_csv(f"{METRICS_DIR}/scalability_sweep_results.csv", index=False)
sweep_df

In [ ]:
plot_scalability_curves(sweep_df)

## Loss Convergence Profiles

In [ ]:
curves = {}
rf_stages = [5, 15, 30, 45, 60] if FAST_MODE else None

for name, model in get_baseline_models().items():
    try:
        train_loss, val_loss = get_learning_curve(name, model, X_train, y_train, X_val, y_val, max_stages=rf_stages if name == "RandomForest" else None)
        if train_loss and val_loss:
            curves[name] = (train_loss, val_loss)
    except MemoryError:
        pass

plot_loss_convergence(curves)

## Feature Importance Divergence

In [ ]:
import gc

importances = {}
for name, model in models_for_analysis.items():
    try:
        fitted = model.fit(X_train, y_train)
        importances[name] = get_feature_importances(name, fitted, X_train.columns)
        importances[name].to_csv(f"{METRICS_DIR}/feature_importance_{name}.csv")
    except MemoryError:
        pass
    finally:
        del model
        gc.collect()

plot_feature_importance_divergence(importances)

## Summary Table

In [ ]:
summary = results_df[["model", "train_time_sec", "inference_latency_ms_per_row", "accuracy", "macro_f1"]].round(4)
summary = summary.merge(cv_df[["model", "cv_accuracy_mean", "cv_macro_f1_mean"]], on="model")
summary.to_csv(f"{METRICS_DIR}/summary.csv", index=False)
summary

## Download Results

In [ ]:
import shutil
from google.colab import files

shutil.make_archive("results", "zip", "results")
files.download("results.zip")